## Do camera calibration

In [12]:
import cv2
import numpy as np
import glob
import os

# --- CONFIGURATION ---
CHECKERBOARD_DIMS = (9, 6) 
SQUARE_SIZE = 0.037  # Meters
CALIB_IMG_DIR = '/Users/bbimali1/Documents/Computer_Vision_Spring_26/module_2_camera_calibration/resources/calibration_images'

def extract_focal_length(calib_img_dir):
    """
    Processes checkerboard images to compute the camera matrix (K), 
    distortion coefficients (D), and averaged focal length in pixels (f).
    """
    # Create a 3D grid of object points (0,0,0), (1,0,0), ...
    objp = np.zeros((CHECKERBOARD_DIMS[0] * CHECKERBOARD_DIMS[1], 3), np.float32)
    objp[:, :2] = np.mgrid[0:CHECKERBOARD_DIMS[0], 0:CHECKERBOARD_DIMS[1]].T.reshape(-1, 2)
    objp = objp * SQUARE_SIZE

    objpoints = [] 
    imgpoints = [] 

    # file search to catch both .jpg and .JPG
    images = glob.glob(os.path.join(calib_img_dir, '*.[jJ][pP][gG]'))
    
    if not images:
        print(f"CRITICAL ERROR: No images found in {calib_img_dir}")
        return None, None, None

    print(f"Processing {len(images)} calibration images...")

    valid_images = 0
    gray_shape = None
    
    for fname in images:
        img = cv2.imread(fname)
        if img is None:
            continue
            
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        gray_shape = gray.shape[::-1]

        # Detect checkerboard corners
        ret, corners = cv2.findChessboardCorners(gray, CHECKERBOARD_DIMS, None)

        if ret:
            valid_images += 1
            objpoints.append(objp)
            
            # Refine corner locations to sub-pixel accuracy
            corners2 = cv2.cornerSubPix(gray, corners, (11, 11), (-1, -1), 
                                        criteria=(cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 30, 0.001))
            imgpoints.append(corners2)
        else:
            print(f"  [SKIP] Pattern not found: {os.path.basename(fname)}")

    # Compute camera matrix
    if len(objpoints) > 0:
        print(f"\nCalibrating with {valid_images} valid images...")
        ret, mtx, dist, rvecs, tvecs = cv2.calibrateCamera(objpoints, imgpoints, gray_shape, None, None)
        
        # Extract focal lengths from the K matrix
        fx = mtx[0, 0]
        fy = mtx[1, 1]
        f_pixels = (fx + fy) / 2.0
        
        print(f"Calibration Complete. Reprojection Error: {ret:.4f} pixels")
        print("="*40)
        print(f"Focal Length X (fx): {fx:.2f} pixels")
        print(f"Focal Length Y (fy): {fy:.2f} pixels")
        print(f"--> Averaged Focal Length (f): {f_pixels:.2f} pixels <--")
        print("="*40 + "\n")
        
        return mtx, dist, f_pixels
    else:
        print("Calibration failed. No valid images found.")
        return None, None, None

if __name__ == "__main__":
    K_matrix, Distortion, f_pixels = extract_focal_length(CALIB_IMG_DIR)

Processing 27 calibration images...

Calibrating with 27 valid images...
Calibration Complete. Reprojection Error: 0.5842 pixels
Focal Length X (fx): 3065.00 pixels
Focal Length Y (fy): 3067.44 pixels
--> Averaged Focal Length (f): 3066.22 pixels <--



## Calculating the coordinates

In [13]:
%matplotlib tk

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

# CALIBRATION INPUTS
B = 1.02  # Baseline in meters (102 cm)

f_pixels = f_pixels  # Focal length in pixels (from calibration)
K = K_matrix
D = Distortion

LEFT_IMG = '/Users/bbimali1/Documents/Computer_Vision_Spring_26/week_8_stereo_camera/IMG/IMG_1875.JPG'
RIGHT_IMG = '/Users/bbimali1/Documents/Computer_Vision_Spring_26/week_8_stereo_camera/IMG/IMG_1874.JPG'

# ROBUST INTERACTION FUNCTION (OpenCV)
def get_user_points(img_path, obj_type, side):
    img = cv2.imread(img_path)
    if img is None:
        print(f"CRITICAL ERROR: Could not find {img_path}")
        return []

    # Apply undistortion to flatten the lens curvature
    img_undistorted = cv2.undistort(img, K, D, None, K)
    clone = img_undistorted.copy()
    points = []

    window_name = f"Select {obj_type} in {side} Image"
    cv2.namedWindow(window_name, cv2.WINDOW_NORMAL)

    def click_event(event, x, y, flags, param):
        if event == cv2.EVENT_LBUTTONDOWN:
            points.append((x, y))
            # Draw a dot and number it on the image for visual feedback
            cv2.circle(clone, (x, y), 8, (0, 0, 255), -1)
            cv2.putText(clone, str(len(points)), (x + 10, y - 10), 
                        cv2.FONT_HERSHEY_SIMPLEX, 2.0, (0, 255, 0), 3)
            cv2.imshow(window_name, clone)
            print(f"  -> Point {len(points)} recorded at: X={x}, Y={y}")

    cv2.imshow(window_name, clone)
    cv2.setMouseCallback(window_name, click_event)
    
    print(f"\n---> {side.upper()} IMAGE: {obj_type} <---")
    print("1. LEFT CLICK to select points in the exact same order.")
    print("2. PRESS ANY KEY on your keyboard (e.g., Spacebar) when finished.")
    
    # Waits indefinitely until we press a key
    cv2.waitKey(0) 
    cv2.destroyAllWindows()
    
    return points

# COORDINATE CALCULATION & VALIDATION
def triangulate_points(left_pts, right_pts, category_name):
    if len(left_pts) != len(right_pts):
        print(f"\nERROR: Mismatch in {category_name}!")
        print(f"You clicked {len(left_pts)} in Left, but {len(right_pts)} in Right.")
        return []

    coords = []
    for i, (lp, rp) in enumerate(zip(left_pts, right_pts)):
        xL, xR = lp[0], rp[0]
        disparity = xL - xR
        
        if disparity <= 0:
            print(f"Warning: {category_name} Item {i+1} has invalid disparity ({disparity}). Skipping.")
            continue
            
        # Core Simple Stereo Math
        Z = (f_pixels * B) / disparity
        X = (xL * Z) / f_pixels
        coords.append((X, Z))
        
    print(f"Successfully calculated 3D coordinates for {len(coords)} {category_name}.")
    return coords

# MAIN EXECUTION
if __name__ == "__main__":
    print("=== STARTING STEREO MAPPING ===")
    
    # Collect Tables
    t_left = get_user_points(LEFT_IMG, "TABLES (Red)", "Left")
    t_right = get_user_points(RIGHT_IMG, "TABLES (Red)", "Right")

    # Collect Chairs
    c_left = get_user_points(LEFT_IMG, "CHAIRS (Blue)", "Left")
    c_right = get_user_points(RIGHT_IMG, "CHAIRS (Blue)", "Right")

    print("\n=== CALCULATING MATH ===")
    # Calculate Coordinates
    tables_2d = triangulate_points(t_left, t_right, "Tables")
    chairs_2d = triangulate_points(c_left, c_right, "Chairs")

    # FINAL 2D PLOT
    print("\nGenerating final plot...")
    plt.figure(figsize=(8, 10))

    if tables_2d:
        tx, tz = zip(*tables_2d)
        plt.scatter(tx, tz, color='red', marker='s', s=150, label='Tables')

    if chairs_2d:
        cx, cz = zip(*chairs_2d)
        plt.scatter(cx, cz, color='blue', marker='o', s=80, label='Chairs')

    # Origin (Camera position)
    plt.scatter(0, 0, color='black', marker='^', s=200, label='Camera (Origin)')
    plt.axhline(0, color='black', linewidth=1) 
    plt.axvline(0, color='black', linewidth=1, linestyle='--') 

    plt.xlabel("X - Lateral Position (meters)", fontsize=12)
    plt.ylabel("Y - Depth into Room (meters)", fontsize=12)
    plt.title("Classroom 2D Floor Plan Layout", fontsize=14, fontweight='bold')
    
    plt.legend()
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.axis('equal') 

    print("Saving plot as 'classroom_map.png'...")
    plt.savefig('classroom_map.png', dpi=300, bbox_inches='tight')
    plt.show()

=== STARTING STEREO MAPPING ===

---> LEFT IMAGE: TABLES (Red) <---
1. LEFT CLICK to select points in the exact same order.
2. PRESS ANY KEY on your keyboard (e.g., Spacebar) when finished.
  -> Point 1 recorded at: X=1328, Y=2073
  -> Point 2 recorded at: X=1168, Y=2209
  -> Point 3 recorded at: X=913, Y=2422
  -> Point 4 recorded at: X=2175, Y=2080
  -> Point 5 recorded at: X=2232, Y=2213
  -> Point 6 recorded at: X=2272, Y=2414
  -> Point 7 recorded at: X=2431, Y=2769

---> RIGHT IMAGE: TABLES (Red) <---
1. LEFT CLICK to select points in the exact same order.
2. PRESS ANY KEY on your keyboard (e.g., Spacebar) when finished.
  -> Point 1 recorded at: X=875, Y=2037
  -> Point 2 recorded at: X=599, Y=2175
  -> Point 3 recorded at: X=161, Y=2379
  -> Point 4 recorded at: X=1668, Y=2032
  -> Point 5 recorded at: X=1620, Y=2190
  -> Point 6 recorded at: X=1579, Y=2370
  -> Point 7 recorded at: X=1421, Y=2746

---> LEFT IMAGE: CHAIRS (Blue) <---
1. LEFT CLICK to select points in the exact 